# I-S101 — 안정화 Dual-Encoder Gated SegResNet

I-S100의 구조는 유지하되 수치 붕괴를 방지하는 재실험이다. I-S100 결과와 checkpoint는 수정하지 않는다.

변경점: learning rate `1e-3 → 3e-4`, AMP 비활성화, gradient clipping `max_norm=1.0`, loss/gradient NaN 즉시 중단, epoch별 gradient norm 기록. 나머지 데이터 split, sampling, 300 epochs, full-volume validation, Top-3 ±5 선택 및 확률맵 평가는 동일하다.

In [1]:
from pathlib import Path
import json, os, signal, subprocess, time
import pandas as pd
from IPython.display import display, clear_output

REPO=Path("/home/stu03/workspace/model")
PYTHON=REPO/".venv/bin/python"
CONFIG=REPO/"configs/experiments/ischemia/I-S101.yaml"
RUN=REPO/"runs/I-segresnet/I-S101"
LOG=RUN/"sequence.log"
PID_FILE=RUN/"sequence.pid"
assert PYTHON.is_file() and CONFIG.is_file()

## 1. 실행 명령 확인

In [2]:
subprocess.run([str(PYTHON),str(REPO/"scripts/ischemia/training/run_i_s_experiment.py"),"--config",str(CONFIG),"--dry-run"],cwd=REPO,check=True)

{
  "experiment": "I-S101",
  "mode": "resume",
  "data_root": "/home/stu03/workspace/datasets/ischemic/SegResNet",
  "results": "/home/stu03/workspace/model/runs/I-segresnet/I-S101",
  "command": [
    "/home/stu03/workspace/model/.venv/bin/python",
    "-m",
    "brainon_ai.models.ischemia.segresnet_hy.train",
    "--config",
    "/home/stu03/workspace/model/configs/experiments/ischemia/I-S101.yaml",
    "--data-root",
    "/home/stu03/workspace/datasets/ischemic/SegResNet",
    "--backup-root",
    "/home/stu03/workspace/model/runs/I-segresnet/I-S101"
  ]
}


CompletedProcess(args=['/home/stu03/workspace/model/.venv/bin/python', '/home/stu03/workspace/model/scripts/ischemia/training/run_i_s_experiment.py', '--config', '/home/stu03/workspace/model/configs/experiments/ischemia/I-S101.yaml', '--dry-run'], returncode=0)

## 2. 실제 batch sanity test

학습은 시작하지 않고 모델 출력 shape와 finite loss만 확인한다.

In [ ]:
subprocess.run([str(PYTHON),"-m","brainon_ai.models.ischemia.segresnet_hy.train","--config",str(CONFIG),"--data-root","/home/stu03/workspace/datasets/ischemic/SegResNet","--backup-root","/tmp/I-S101_sanity","--sanity-only"],cwd=REPO,env={**os.environ,"PYTHONPATH":str(REPO/"src")},check=True)

## 3. 학습 시작/재개

기존 `checkpoint_latest.pth`가 있으면 자동 재개한다.

In [3]:
def process_is_alive(pid: int) -> bool:
    """Treat a reaped/zombie PID as stopped; PID files survive notebook restarts."""
    try:
        state = Path(f"/proc/{pid}/stat").read_text().split()[2]
        return state != "Z"
    except (FileNotFoundError, IndexError):
        return False

RUN.mkdir(parents=True,exist_ok=True)
if PID_FILE.is_file():
    old=int(PID_FILE.read_text())
    if process_is_alive(old):
        raise RuntimeError(f"이미 실행 중: PID={old}")
    print(f"stale PID file ignored: {old}")
handle=LOG.open("a",buffering=1)
cmd=[str(PYTHON),str(REPO/"scripts/ischemia/training/run_i_s_experiment.py"),"--config",str(CONFIG)]
proc=subprocess.Popen(cmd,cwd=REPO,stdout=handle,stderr=subprocess.STDOUT,start_new_session=True)
PID_FILE.write_text(str(proc.pid)); print(f"started PID={proc.pid} | {LOG}")

stale PID file ignored: 2327795
started PID=2351507 | /home/stu03/workspace/model/runs/I-segresnet/I-S101/sequence.log


## 4. 실시간 진행률

10초마다 갱신한다. 모니터 셀을 중단해도 background 학습은 계속된다. 완료·실패 상태에서는 자동 종료한다.

In [4]:
def alive():
    if not PID_FILE.is_file(): return False
    try:
        pid=int(PID_FILE.read_text())
        return Path(f"/proc/{pid}/stat").read_text().split()[2] != "Z"
    except (FileNotFoundError, IndexError, ValueError):
        return False
try:
    while True:
        progress=json.loads((RUN/"progress.json").read_text()) if (RUN/"progress.json").is_file() else {"status":"waiting"}
        clear_output(wait=True); print(time.strftime("Updated: %Y-%m-%d %H:%M:%S")); display(pd.DataFrame([progress]))
        numeric=RUN/"logs/numerics_log.csv"
        if numeric.is_file(): display(pd.read_csv(numeric).tail(10))
        if LOG.is_file(): print("\n--- latest log ---\n"+"".join(LOG.read_text(errors="replace").splitlines(True)[-20:]))
        terminal=progress.get("status") in {"completed","failed_nonfinite_loss","failed_nonfinite_gradient"}
        if terminal or not alive(): print("\n실행 종료. 마지막 상태입니다."); break
        time.sleep(10)
except KeyboardInterrupt: print("모니터만 중단했습니다. background 학습은 계속됩니다.")

Updated: 2026-09-16 12:22:25


,status,current_epoch,total_epochs,progress_percent,eta_seconds,full_macro_dice,best_full_macro_dice,retained_epoch_checkpoints,max_gradient_norm,gradient_clip_max_norm,amp_enabled,anchors,neighborhood_epochs,selected_epoch,selected_macro_dice,selected_lesion_recall
0,completed,300,300,100.0,0,0.761661,0.782045,33,4.51392,1.0,False,"[90, 170, 60]","[55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 8...",90,0.776154,0.655319


,epoch,max_gradient_norm,amp_scale
290,291,1.369666,NaN
291,292,3.090832,NaN
292,293,8.680780,NaN
293,294,1.220927,NaN
294,295,3.543671,NaN
295,296,3.693396,NaN
296,297,3.121521,NaN
297,298,4.479797,NaN
298,299,1.310501,NaN
299,300,4.513920,NaN



--- latest log ---
      "lesion_detection_rate": 0.6340425531914894,
      "checkpoint": "/home/stu03/workspace/model/runs/I-segresnet/I-S101/checkpoints/epoch_checkpoints/checkpoint_epoch_0088.pth",
      "evaluation": "/home/stu03/workspace/model/runs/I-segresnet/I-S101/selection_evaluation/epoch_0088"
    },
    {
      "epoch": 170,
      "macro_dice_positive": 0.7688284741942277,
      "lesion_detection_rate": 0.6468085106382979,
      "checkpoint": "/home/stu03/workspace/model/runs/I-segresnet/I-S101/checkpoints/epoch_checkpoints/checkpoint_epoch_0170.pth",
      "evaluation": "/home/stu03/workspace/model/runs/I-segresnet/I-S101/selection_evaluation/epoch_0170"
    }
  ],
  "selected": {
    "epoch": 90,
    "macro_dice_positive": 0.7761539214317155,
    "lesion_detection_rate": 0.6553191489361702,
    "checkpoint": "/home/stu03/workspace/model/runs/I-segresnet/I-S101/checkpoints/epoch_checkpoints/checkpoint_epoch_0090.pth",
    "evaluation": "/home/stu03/workspace/model/runs/I

## 5. 중단 — 필요할 때만 실행

In [ ]:
pid=int(PID_FILE.read_text()); os.killpg(pid,signal.SIGTERM); print(f"stop requested: {pid}")

## 6. 최종 결과

In [5]:
selection=RUN/"final_selection.json"; sweep=RUN/"final_metrics/threshold_sweep.csv"
if not selection.is_file(): print("최종 선택 결과가 아직 없습니다.")
else:
    print(json.dumps(json.loads(selection.read_text())["selected"],ensure_ascii=False,indent=2))
    if sweep.is_file():
        df=pd.read_csv(sweep); display(df.sort_values(["lesion_f2","macro_dice_positive"],ascending=False).head(10))

{
  "epoch": 90,
  "macro_dice_positive": 0.7761539214317155,
  "lesion_detection_rate": 0.6553191489361702,
  "checkpoint": "/home/stu03/workspace/model/runs/I-segresnet/I-S101/checkpoints/epoch_checkpoints/checkpoint_epoch_0090.pth",
  "evaluation": "/home/stu03/workspace/model/runs/I-segresnet/I-S101/selection_evaluation/epoch_0090"
}


,threshold,macro_dice_positive,voxel_recall,voxel_precision,lesion_recall,lesion_precision,lesion_f1,lesion_f2,small_lesion_recall,small_lesion_dice,fp_per_scan,negative_case_fp_rate,detected_lesions,total_gt_lesions,false_positive_lesions
5,0.35,0.778909,0.776584,0.829794,0.676596,0.753247,0.712867,0.690652,0.615385,0.397391,1.500000,0.0,159,235,57
2,0.20,0.773789,0.806217,0.784787,0.693617,0.674699,0.684027,0.689749,0.635897,0.388267,2.131579,0.0,163,235,81
3,0.25,0.777727,0.796148,0.804551,0.685106,0.706383,0.695582,0.689259,0.625641,0.396398,1.815789,0.0,161,235,69
4,0.30,0.778464,0.786432,0.816969,0.680851,0.724576,0.702033,0.689169,0.620513,0.399188,1.710526,0.0,160,235,65
6,0.40,0.780002,0.767561,0.843951,0.672340,0.752174,0.710020,0.686922,0.610256,0.396007,1.500000,0.0,158,235,57
1,0.15,0.769633,0.821564,0.761896,0.697872,0.637795,0.666483,0.684968,0.641026,0.386492,2.421053,0.0,164,235,92
10,0.60,0.771355,0.724078,0.887611,0.655319,0.809091,0.724132,0.681213,0.589744,0.385776,1.105263,0.0,154,235,42
7,0.45,0.778177,0.754884,0.855285,0.663830,0.757709,0.707670,0.680697,0.600000,0.393334,1.447368,0.0,156,235,55
9,0.55,0.773808,0.734744,0.875393,0.655319,0.781818,0.713001,0.677235,0.589744,0.385220,1.263158,0.0,154,235,48
8,0.50,0.776154,0.745386,0.863913,0.655319,0.752174,0.700414,0.672642,0.589744,0.387113,1.500000,0.0,154,235,57
